# Evaluation Anchors and Bootstrap CIs

## 0. Install/check packages

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "transformers": "transformers",
}
missing = [pip_name for module_name, pip_name in required.items() if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


## 1. Setup

In [ ]:
# Portable project-path setup.
# Override any of these by exporting the matching env var before launching
# the notebook (e.g. `export PROJECT_ROOT=/path/to/VulnerableCancerPatients`):
#   PROJECT_ROOT  - root of the experiment tree
#   SCRIPTS_DIR   - shared helper modules (default: PROJECT_ROOT/scripts)
#   DATA_DIR      - shared data directory (default: PROJECT_ROOT/data)
import os
import sys
from pathlib import Path

FOLDER_NAME = "06_EvaluationAnchors_Bootstrap"
COLAB_DEFAULT = Path("/content/drive/MyDrive/NLP_Projects/VulnerableCancerPatients")


def _resolve_project_root() -> Path:
    env = os.environ.get("PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=False)
        if COLAB_DEFAULT.exists():
            return COLAB_DEFAULT.resolve()
    except ImportError:
        pass
    cwd = Path.cwd().resolve()
    if cwd.name == FOLDER_NAME:
        return cwd.parent
    if (cwd / FOLDER_NAME).exists():
        return cwd
    return cwd


PROJECT_ROOT = _resolve_project_root()
BASE_DIR = PROJECT_ROOT / FOLDER_NAME if (PROJECT_ROOT / FOLDER_NAME).exists() else PROJECT_ROOT
SCRIPTS_DIR = Path(os.environ.get("SCRIPTS_DIR", PROJECT_ROOT / "scripts")).expanduser().resolve()
DATA_DIR = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print("Project root:", PROJECT_ROOT)
print("Base:", BASE_DIR)
print("Scripts:", SCRIPTS_DIR, "(exists)" if SCRIPTS_DIR.exists() else "(MISSING)")
print("Data:", DATA_DIR, "(exists)" if DATA_DIR.exists() else "(MISSING)")
print("Outputs:", OUTPUT_DIR)


In [ ]:
import os

print(f"Listing contents of PROJECT_ROOT: {PROJECT_ROOT}")

if not PROJECT_ROOT.exists():
    print(f"Error: PROJECT_ROOT does not exist at {PROJECT_ROOT}")
elif not PROJECT_ROOT.is_dir():
    print(f"Error: PROJECT_ROOT is not a directory at {PROJECT_ROOT}")
else:
    for root, dirs, files in os.walk(PROJECT_ROOT):
        level = root.replace(str(PROJECT_ROOT), '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')


## 2. Load shared data and split

In [ ]:
from data_utils import (
    EMOTION_LABELS_3,
    EMOTION_PROB_COLS_3,
    HEOR_SUBSCALES,
    RAW_LLM_PROB_COLS_4,
    prepare_annotation_frame,
)
from metrics import (
    DEFAULT_BOOTSTRAP_N,
    bootstrap_classification_metric_rows,
    classification_metrics,
    multiclass_brier_score,
    one_hot,
    paired_delta_ci,
    soft_cross_entropy,
)

df = prepare_annotation_frame(ANNOTATION_PATH, SPLIT_PATH)
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("Shape:", df.shape)
print(df["split"].value_counts().sort_index())
print("Bootstrap n:", DEFAULT_BOOTSTRAP_N)


## 3. Study 2 Brier and soft-CE anchors

In [ ]:
soft_cols = EMOTION_PROB_COLS_3
y_soft = test_df[soft_cols].astype(float).to_numpy()
n = len(test_df)

majority_class = int(np.argmax(y_soft.sum(axis=0)))
majority_probs = np.zeros_like(y_soft)
majority_probs[:, majority_class] = 1.0
uniform_probs = np.full_like(y_soft, 1.0 / y_soft.shape[1])
llm_argmax_probs = one_hot(np.argmax(y_soft, axis=1), y_soft.shape[1])

def bootstrap_anchor_ci(target, probs, n_boot=DEFAULT_BOOTSTRAP_N, seed=42):
    rng = np.random.default_rng(seed)
    brier_samples = []
    soft_ce_samples = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(target), len(target))
        brier_samples.append(multiclass_brier_score(target[idx], probs[idx]))
        soft_ce_samples.append(soft_cross_entropy(target[idx], probs[idx]))
    return {
        "brier_ci_low": float(np.percentile(brier_samples, 2.5)),
        "brier_ci_high": float(np.percentile(brier_samples, 97.5)),
        "soft_cross_entropy_ci_low": float(np.percentile(soft_ce_samples, 2.5)),
        "soft_cross_entropy_ci_high": float(np.percentile(soft_ce_samples, 97.5)),
    }

anchor_rows = []
for name, probs in [
    ("majority_class", majority_probs),
    ("uniform", uniform_probs),
    ("llm_argmax_oracle", llm_argmax_probs),
]:
    ci = bootstrap_anchor_ci(y_soft, probs)
    anchor_rows.append({
        "anchor": name,
        "majority_class_id": majority_class if name == "majority_class" else np.nan,
        "majority_class_name": EMOTION_LABELS_3[majority_class] if name == "majority_class" else "",
        "brier_against_llm_distribution": multiclass_brier_score(y_soft, probs),
        "soft_cross_entropy_against_llm_distribution": soft_cross_entropy(y_soft, probs),
        **ci,
        "n_boot": DEFAULT_BOOTSTRAP_N,
        "ci_method": "nonparametric_percentile",
        "n": n,
    })

anchors = pd.DataFrame(anchor_rows)
anchors.to_csv(OUTPUT_DIR / "study2_brier_anchors.csv", index=False)
anchors


## 4. Four-class JSD between human labels and LLM distributions

In [ ]:
def normalize_rows(arr):
    arr = np.asarray(arr, dtype=float)
    sums = arr.sum(axis=1, keepdims=True)
    return np.divide(arr, sums, out=np.full_like(arr, 1.0 / arr.shape[1]), where=sums != 0)

def kl_div(p, q, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * np.log(p / q), axis=1)

def js_divergence(p, q):
    p = normalize_rows(p)
    q = normalize_rows(q)
    m = 0.5 * (p + q)
    return 0.5 * kl_div(p, m) + 0.5 * kl_div(q, m)

human4 = pd.get_dummies(
    df["intensity"].astype(int).map({-2: "very_negative", -1: "negative", 0: "neutral", 1: "positive"})
).reindex(columns=["very_negative", "negative", "neutral", "positive"], fill_value=0).to_numpy(float)

llm4 = df[RAW_LLM_PROB_COLS_4].astype(float).to_numpy()
jsd_values = js_divergence(human4, llm4)

rng = np.random.default_rng(42)
mean_samples = []
median_samples = []
for _ in range(DEFAULT_BOOTSTRAP_N):
    idx = rng.integers(0, len(jsd_values), len(jsd_values))
    mean_samples.append(float(np.mean(jsd_values[idx])))
    median_samples.append(float(np.median(jsd_values[idx])))

jsd_summary = pd.DataFrame([{
    "comparison": "human_4class_onehot_vs_llm_4class_distribution",
    "mean_jsd": float(np.mean(jsd_values)),
    "mean_jsd_ci_low": float(np.percentile(mean_samples, 2.5)),
    "mean_jsd_ci_high": float(np.percentile(mean_samples, 97.5)),
    "median_jsd": float(np.median(jsd_values)),
    "median_jsd_ci_low": float(np.percentile(median_samples, 2.5)),
    "median_jsd_ci_high": float(np.percentile(median_samples, 97.5)),
    "sd_jsd": float(np.std(jsd_values, ddof=1)),
    "n_boot": DEFAULT_BOOTSTRAP_N,
    "ci_method": "nonparametric_percentile",
    "n": int(len(jsd_values)),
}])
jsd_summary.to_csv(OUTPUT_DIR / "study2_4class_jsd.csv", index=False)
jsd_summary


## 5. Sequence truncation analysis at 384 tokens

In [ ]:
from transformers import AutoTokenizer

MAX_LENGTH = 384
tokenizer = AutoTokenizer.from_pretrained("albert-base-v2", use_fast=True)

lengths = []
for text in df["posts"].astype(str).tolist():
    lengths.append(len(tokenizer.encode(text, add_special_tokens=True, truncation=False)))

trunc_df = df[["source_row", "split", "human_emotion_3class", "ai_high_need_flag", *HEOR_SUBSCALES]].copy()
trunc_df["token_count_albert"] = lengths
trunc_df["truncated_at_384"] = trunc_df["token_count_albert"] > MAX_LENGTH
trunc_df.to_csv(OUTPUT_DIR / "sequence_truncation_by_row.csv", index=False)

def truncation_summary_row(group_name, group, n_boot=DEFAULT_BOOTSTRAP_N, seed=42):
    token_values = group["token_count_albert"].to_numpy()
    trunc_values = group["truncated_at_384"].astype(float).to_numpy()
    rng = np.random.default_rng(seed)
    pct_samples = []
    median_samples = []
    p95_samples = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(group), len(group))
        pct_samples.append(float(trunc_values[idx].mean() * 100))
        median_samples.append(float(np.median(token_values[idx])))
        p95_samples.append(float(np.quantile(token_values[idx], 0.95)))
    return {
        "group": group_name,
        "n": len(group),
        "n_truncated": int(group["truncated_at_384"].sum()),
        "pct_truncated": float(group["truncated_at_384"].mean() * 100),
        "pct_truncated_ci_low": float(np.percentile(pct_samples, 2.5)),
        "pct_truncated_ci_high": float(np.percentile(pct_samples, 97.5)),
        "median_tokens": float(group["token_count_albert"].median()),
        "median_tokens_ci_low": float(np.percentile(median_samples, 2.5)),
        "median_tokens_ci_high": float(np.percentile(median_samples, 97.5)),
        "p95_tokens": float(group["token_count_albert"].quantile(0.95)),
        "p95_tokens_ci_low": float(np.percentile(p95_samples, 2.5)),
        "p95_tokens_ci_high": float(np.percentile(p95_samples, 97.5)),
        "n_boot": n_boot,
        "ci_method": "nonparametric_percentile",
    }

summary_rows = []
for group_name, group in [("all", trunc_df)]:
    summary_rows.append(truncation_summary_row(group_name, group))
for label, group in trunc_df.groupby("human_emotion_3class"):
    summary_rows.append(truncation_summary_row(f"human_emotion={label}", group))
for flag, group in trunc_df.groupby("ai_high_need_flag"):
    summary_rows.append(truncation_summary_row(f"ai_high_need={flag}", group))
for subscale in HEOR_SUBSCALES:
    for level, group in trunc_df.groupby(subscale):
        summary_rows.append(truncation_summary_row(f"{subscale}={level}", group))

trunc_summary = pd.DataFrame(summary_rows)
trunc_summary.to_csv(OUTPUT_DIR / "sequence_truncation_analysis.csv", index=False)
trunc_summary.head(20)


## 6. Load prediction files when available

In [ ]:
PREDICTION_FILES = {
    "hard_label_regular": PROJECT_ROOT / "01_HardLabelBaseline" / "outputs" / "hard_label_regular_predictions_test.csv",
    "hard_label_augmented": PROJECT_ROOT / "01_HardLabelBaseline" / "outputs" / "hard_label_augmented_predictions_test.csv",
    "hard_label_archived_regular_logistic_regression": PROJECT_ROOT / "01_HardLabelBaseline" / "outputs" / "hard_label_archived_regular_logistic_regression_predictions_test.csv",
    "hard_label_archived_regular_random_forest": PROJECT_ROOT / "01_HardLabelBaseline" / "outputs" / "hard_label_archived_regular_random_forest_predictions_test.csv",
    "hard_label_archived_regular_lightgbm": PROJECT_ROOT / "01_HardLabelBaseline" / "outputs" / "hard_label_archived_regular_lightgbm_predictions_test.csv",
    "hard_label_archived_regular_gru": PROJECT_ROOT / "01_HardLabelBaseline" / "outputs" / "hard_label_archived_regular_gru_predictions_test.csv",
    "llm_argmax_regular": PROJECT_ROOT / "02_LLMArgmaxHardLabelControl" / "outputs" / "llm_argmax_regular_predictions_test.csv",
    "llm_argmax_augmented": PROJECT_ROOT / "02_LLMArgmaxHardLabelControl" / "outputs" / "llm_argmax_augmented_predictions_test.csv",
    "soft_label_regular": PROJECT_ROOT / "03_SoftLabelSupervision" / "outputs" / "soft_label_regular_predictions_test.csv",
    "soft_label_augmented": PROJECT_ROOT / "03_SoftLabelSupervision" / "outputs" / "soft_label_augmented_predictions_test.csv",
}

prediction_frames = {}
missing = []
for name, path in PREDICTION_FILES.items():
    if path.exists():
        prediction_frames[name] = pd.read_csv(path)
        print("Loaded", name, path)
    else:
        missing.append({"condition": name, "path": str(path)})

missing_predictions = pd.DataFrame(missing)
missing_predictions.to_csv(OUTPUT_DIR / "missing_prediction_files.csv", index=False)
missing_predictions


## 7. Study 2 per-class metrics with 1000 bootstrap CIs

In [ ]:
metric_rows = []
ci_frames = []
for condition, pred in prediction_frames.items():
    prob_cols = [f"prob_{name.lower()}" for name in EMOTION_LABELS_3]
    if not set(["y_true_human_emotion", "y_pred", *prob_cols]).issubset(pred.columns):
        print("Skipping incomplete prediction file:", condition)
        continue
    y_true = pred["y_true_human_emotion"].astype(int).to_numpy()
    y_pred = pred["y_pred"].astype(int).to_numpy()
    y_prob = pred[prob_cols].astype(float).to_numpy()
    soft_cols_pred = [f"llm_target_{name.lower()}" for name in EMOTION_LABELS_3]
    soft_target = pred[soft_cols_pred].astype(float).to_numpy() if set(soft_cols_pred).issubset(pred.columns) else None
    m = classification_metrics(y_true, y_pred, y_prob, EMOTION_LABELS_3, soft_target=soft_target)
    m.update({"condition": condition})
    metric_rows.append(m)
    ci = bootstrap_classification_metric_rows(
        y_true,
        y_pred,
        y_prob,
        class_names=EMOTION_LABELS_3,
        soft_target=soft_target,
        n_boot=1000,
        seed=42,
    )
    ci.insert(0, "condition", condition)
    ci_frames.append(ci)

study2_metrics = pd.DataFrame(metric_rows)
study2_metrics.to_csv(OUTPUT_DIR / "study2_per_class_metrics.csv", index=False)
if ci_frames:
    study2_ci = pd.concat(ci_frames, ignore_index=True)
else:
    study2_ci = pd.DataFrame()
study2_ci.to_csv(OUTPUT_DIR / "study2_per_class_metrics_ci.csv", index=False)
study2_metrics


## 8. Study 2 paired delta CIs

In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score

def weighted_f1_metric(y_true, y_pred):
    return f1_score(y_true, y_pred, average="weighted", zero_division=0)

def macro_f1_metric(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

def balanced_accuracy_metric(y_true, y_pred):
    return balanced_accuracy_score(y_true, y_pred)

delta_metrics = [
    ("weighted_f1", weighted_f1_metric),
    ("macro_f1", macro_f1_metric),
    ("balanced_accuracy", balanced_accuracy_metric),
]

delta_pairs = [
    ("soft_minus_hard_regular", "soft_label_regular", "hard_label_regular"),
    ("soft_minus_hard_augmented", "soft_label_augmented", "hard_label_augmented"),
    ("soft_minus_llm_argmax_regular", "soft_label_regular", "llm_argmax_regular"),
    ("soft_minus_llm_argmax_augmented", "soft_label_augmented", "llm_argmax_augmented"),
    ("augmented_minus_regular_hard", "hard_label_augmented", "hard_label_regular"),
    ("augmented_minus_regular_soft", "soft_label_augmented", "soft_label_regular"),
    ("augmented_minus_regular_llm_argmax", "llm_argmax_augmented", "llm_argmax_regular"),
]

delta_rows = []
for label, a, b in delta_pairs:
    if a not in prediction_frames or b not in prediction_frames:
        continue
    left = prediction_frames[a][["source_row", "y_true_human_emotion", "y_pred"]].rename(columns={"y_pred": "pred_a"})
    right = prediction_frames[b][["source_row", "y_pred"]].rename(columns={"y_pred": "pred_b"})
    merged = left.merge(right, on="source_row", validate="one_to_one")
    for metric_name, metric_fn in delta_metrics:
        delta, lo, hi = paired_delta_ci(
            merged["y_true_human_emotion"].to_numpy(),
            merged["pred_a"].to_numpy(),
            merged["pred_b"].to_numpy(),
            metric_fn,
            n_boot=1000,
            seed=42,
        )
        delta_rows.append({
            "comparison": label,
            "condition_a": a,
            "condition_b": b,
            "metric": f"{metric_name}_delta_a_minus_b",
            "delta": delta,
            "ci_low": lo,
            "ci_high": hi,
            "n_boot": 1000,
            "n": len(merged),
        })

study2_deltas = pd.DataFrame(delta_rows)
study2_deltas.to_csv(OUTPUT_DIR / "study2_delta_bootstrap_ci.csv", index=False)
study2_deltas


## 9. Study 1 paired delta CIs for HEOR MTL conditions

In [ ]:
from sklearn.metrics import cohen_kappa_score, f1_score, mean_absolute_error

def load_heor_prediction(name):
    path = PROJECT_ROOT / "04_HEOR_MTL" / "outputs" / f"{name}_predictions_test.csv"
    return pd.read_csv(path) if path.exists() else None

def weighted_f1_metric(y_true, y_pred):
    return f1_score(y_true, y_pred, average="weighted", zero_division=0)

def qwk_metric(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

def mae_metric(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

heor_delta_metrics = [
    ("weighted_f1", weighted_f1_metric),
    ("qwk", qwk_metric),
    ("mae", mae_metric),
]

heor_conditions = [
    "heor_mtl_subscales",
    "heor_mtl_subscales_rc",
    "heor_mtl_subscales_rc_role_cap",
    "heor_mtl_subscales_rc_r0_setup",
]
heor_preds = {name: load_heor_prediction(name) for name in heor_conditions}
heor_preds = {name: frame for name, frame in heor_preds.items() if frame is not None}

heor_delta_rows = []
if "heor_mtl_subscales" in heor_preds:
    base = heor_preds["heor_mtl_subscales"]
    for compare_name, compare in heor_preds.items():
        if compare_name == "heor_mtl_subscales":
            continue
        for task in HEOR_SUBSCALES:
            cols = ["source_row", f"{task}_true", f"{task}_pred"]
            if not set(cols).issubset(base.columns) or not set(cols).issubset(compare.columns):
                continue
            a = compare[cols].rename(columns={f"{task}_pred": "pred_a", f"{task}_true": "true"})
            b = base[["source_row", f"{task}_pred"]].rename(columns={f"{task}_pred": "pred_b"})
            merged = a.merge(b, on="source_row", validate="one_to_one")
            for metric_name, metric_fn in heor_delta_metrics:
                delta, lo, hi = paired_delta_ci(
                    merged["true"].to_numpy(),
                    merged["pred_a"].to_numpy(),
                    merged["pred_b"].to_numpy(),
                    metric_fn,
                    n_boot=1000,
                    seed=42,
                )
                heor_delta_rows.append({
                    "comparison": f"{compare_name}_minus_subscales",
                    "task": task,
                    "metric": f"{metric_name}_delta",
                    "delta": delta,
                    "ci_low": lo,
                    "ci_high": hi,
                    "n_boot": 1000,
                    "n": len(merged),
                })

study1_deltas = pd.DataFrame(heor_delta_rows)
study1_deltas.to_csv(OUTPUT_DIR / "study1_delta_bootstrap_ci.csv", index=False)
study1_deltas
